# Lambda Functions, `map`, `filter` and `reduce` — Full-Time Edition

## 🎯 Learning objectives

By the end of this notebook, you should be able to:

- explain what a `lambda` function is;
- decide when a short operation is a good candidate for `lambda`;
- use a `lambda` as the `key` in `sorted()`;
- use `map()` to transform every element of an iterable;
- use `filter()` to keep only the elements that satisfy a condition;
- use `reduce()` to combine many values into one accumulated result;
- recognize the relationship between `map()` / `filter()` and comprehensions;
- decide when a named function created with `def` is clearer than a lambda.

> **Class rhythm**
>
> Short explanation → example → quick check.
>
> The examples deliberately use contexts different from the Lab.


## 1. Lambda functions

A `lambda` is a **small anonymous function** written as a single expression.

```python
lambda arguments: expression
```

It is most useful when a short function is needed **briefly**, often as an argument to another function.


### From `def` to `lambda`

These two functions do the same computation:


In [ ]:
def apply_discount(price):
    return price * 0.9

apply_discount_lambda = lambda price: price * 0.9

print(apply_discount(80))
print(apply_discount_lambda(80))


A lambda is limited to **one expression**, but that expression can still call other functions.

The more important design question is not only:

> **Can I write this as a lambda?**

but also:

> **Should I write this as a lambda?**

A useful rule of thumb:

```text
Do I need to reuse or clearly name this operation?
        │
   ┌────┴────┐
  YES        NO
   │          │
  def     Is it a simple
          expression?
             │
        ┌────┴────┐
       YES        NO
        │          │
      lambda      def
```

If a lambda becomes hard to read, prefer a normal `def`.


### ⚡ Quick check — complete the lambda

Complete the lambda so it converts Celsius to Fahrenheit.


In [ ]:
to_fahrenheit = lambda celsius: __________

# Expected result:
# to_fahrenheit(25) -> 77.0


### Multiple arguments

A lambda can receive more than one argument, just like a regular function.


In [ ]:
travel_cost = lambda nights, price_per_night: nights * price_per_night

travel_cost(3, 95)


### ⚡ Quick check — two arguments

Create a lambda that receives `distance_km` and `price_per_km` and returns the trip cost.


In [ ]:
trip_cost = lambda __________

# Expected:
# trip_cost(12, 1.5) -> 18.0


## 2. Using `lambda` with `sorted()`

One of the clearest uses of lambda is telling `sorted()` **which value should control the ordering**.

```python
sorted(iterable, key=function, reverse=False)
```

The `key` function receives **one element at a time** and returns the value Python should use for sorting.


In [ ]:
destinations = [
    ("Lisbon", 120),
    ("Rome", 150),
    ("Berlin", 95),
    ("Athens", 135)
]

sorted_destinations = sorted(
    destinations,
    key=lambda destination: destination[1]
)

sorted_destinations


Think about the question:

> **Which part of each element controls the order?**

Here, `destination[1]` is the daily cost, so it becomes the sorting key.


### ⚡ Quick check — sort by rating

Sort the courses from highest to lowest rating.


In [ ]:
courses = [
    ("Python", 4.7),
    ("SQL", 4.9),
    ("Tableau", 4.5)
]

sorted_courses = sorted(
    courses,
    key=lambda course: __________,
    reverse=True
)

sorted_courses


# 3. A useful mental model

Before writing code, ask what you want to do with the iterable:

| Tool | Main idea | Typical shape |
|---|---|---|
| `map()` | **transform** every element | iterable → iterable |
| `filter()` | **select** matching elements | iterable → smaller iterable |
| `reduce()` | **accumulate** everything | iterable → one result |
| `sorted(..., key=...)` | **order** elements | iterable → sorted list |

`lambda` is often the short function supplied to these tools.


## 4. `map()` — transform every element

`map()` applies a function to **every element** of an iterable.

```python
map(function, iterable)
```

It returns a `map` object, so we often wrap it with `list()` when we want to inspect the results.


In [ ]:
temperatures_c = [12, 18, 25, 31]

temperatures_f = list(
    map(lambda c: c * 9 / 5 + 32, temperatures_c)
)

temperatures_f


Think of `map()` as:

> **same number of elements, transformed values**


### Connection with list comprehensions

The same transformation can often be written as a comprehension:


In [ ]:
temperatures_f_map = list(
    map(lambda c: c * 9 / 5 + 32, temperatures_c)
)

temperatures_f_comp = [
    c * 9 / 5 + 32
    for c in temperatures_c
]

print(temperatures_f_map)
print(temperatures_f_comp)


Both are valid.

A comprehension is often very readable for a simple transformation.

`map()` is particularly natural when we already have a function that performs the transformation.


In [ ]:
def to_fahrenheit(celsius):
    return celsius * 9 / 5 + 32

temperatures_f = list(map(to_fahrenheit, temperatures_c))

temperatures_f


### ⚡ Quick check — transform with `map()`

Convert each duration from minutes to hours.


In [ ]:
durations_minutes = [30, 75, 120, 150]

hours = list(
    map(lambda minutes: __________, durations_minutes)
)

# Expected:
# [0.5, 1.25, 2.0, 2.5]


## 5. `filter()` — keep matching elements

`filter()` keeps only the elements for which a function returns `True`.

```python
filter(function, iterable)
```

It also returns an iterator, so we usually wrap it with `list()` when we want a list.


In [ ]:
ratings = [4.8, 3.9, 4.6, 2.8, 4.2]

high_ratings = list(
    filter(lambda rating: rating >= 4.5, ratings)
)

high_ratings


Think of `filter()` as:

> **same kind of elements, but only some remain**


### Connection with comprehensions

This:


In [ ]:
high_ratings_filter = list(
    filter(lambda rating: rating >= 4.5, ratings)
)


is conceptually equivalent to:


In [ ]:
high_ratings_comp = [
    rating
    for rating in ratings
    if rating >= 4.5
]


### ⚡ Quick check — filter short trips

Keep only trips shorter than 5 hours.


In [ ]:
trip_hours = [2.5, 6.0, 4.25, 8.5, 3.0]

short_trips = list(
    filter(lambda hours: __________, trip_hours)
)

# Expected:
# [2.5, 4.25, 3.0]


## 6. `reduce()` — combine many values into one result

`reduce()` is different from `map()` and `filter()`.

It usually turns an iterable into **one accumulated result**.

Before using `reduce()`, let's build the same pattern ourselves with a regular `for` loop.


### 6.1 Start with a normal accumulator

You already know this pattern:


In [ ]:
numbers = [2, 4, 7, 3]

accumulator = 0

for number in numbers:
    accumulator = accumulator + number

print(accumulator)


Step by step:

```text
accumulator = 0

number = 2  → accumulator = 0  + 2 = 2
number = 4  → accumulator = 2  + 4 = 6
number = 7  → accumulator = 6  + 7 = 13
number = 3  → accumulator = 13 + 3 = 16
```

There are two clearly different roles:

```text
accumulator → result obtained so far
number      → next element from the iterable
```


### 6.2 Extract the operation into a function

The operation performed inside the loop is:

```python
accumulator + next_value
```

We can put that operation inside a regular function:


In [ ]:
def addition(accumulator, next_value):
    return accumulator + next_value

print(addition(2, 4))


Conceptually, our loop is doing something like:

```python
accumulator = addition(accumulator, next_value)
```

again and again for every element in the iterable.


### 6.3 What does `reduce()` automate?

`reduce()` automates this **accumulation pattern**.

It lives in `functools`:


In [ ]:
from functools import reduce


We can now write:


In [ ]:
numbers = [2, 4, 7, 3]

total = reduce(addition, numbers, 0)

print(total)


The third argument, `0`, is the **initial accumulator value**.

So this:

```python
reduce(addition, numbers, 0)
```

follows the same idea as:

```python
accumulator = 0

for next_value in numbers:
    accumulator = addition(accumulator, next_value)
```

This is the key connection between the `for` loop and `reduce()`.


### 6.4 The `reduce()` contract

The function passed to `reduce()` must receive **two arguments**.

By definition:

```text
first parameter  ← current accumulator
second parameter ← next element from the iterable
```

The **names do not matter**.

Only the **position** matters.


In [ ]:
def operation(x, y):
    return x + y

reduce(operation, [2, 4, 7, 3], 0)


In the function above:

```text
x → accumulator
y → next value
```

because `x` is the **first parameter** and `y` is the **second parameter**.

We could even use misleading names:


In [ ]:
def operation(next_value, accumulator):
    return next_value + accumulator

reduce(operation, [2, 4, 7, 3], 0)


Despite the names:

```text
next_value   → receives the accumulator
accumulator  → receives the next iterable element
```

because `reduce()` follows the **parameter positions**, not the variable names.

For clarity, we will normally use meaningful names:

```python
def operation(accumulator, next_value):
    ...
```


### 6.5 Step by step with an initial value

For:

```python
numbers = [2, 4, 7, 3]
```

and:

```python
reduce(addition, numbers, 0)
```

the calls are conceptually:


```text
Initial accumulator = 0

addition(0, 2)  → 2
addition(2, 4)  → 6
addition(6, 7)  → 13
addition(13, 3) → 16
```

The value returned by each call becomes the **accumulator** for the next call.


### 6.6 Now replace the named function with a lambda

Once the two roles are clear, the named function:


In [ ]:
def addition(accumulator, next_value):
    return accumulator + next_value


can be replaced by:


In [ ]:
numbers = [2, 4, 7, 3]

total = reduce(
    lambda accumulator, next_value: accumulator + next_value,
    numbers,
    0
)

print(total)


Read the lambda by position:

```text
lambda accumulator, next_value: accumulator + next_value
       ↑            ↑
       │            └── second argument supplied by reduce()
       │
       └── first argument supplied by reduce()
           = current accumulator
```


### 6.7 What if we omit the initial value?

The initial value is optional.

If we write:


In [ ]:
numbers = [2, 4, 7, 3]

total = reduce(
    lambda accumulator, next_value: accumulator + next_value,
    numbers
)

print(total)


then `reduce()` uses the **first element** as the initial accumulator and the **second element** as the first `next_value`.

```text
numbers = [2, 4, 7, 3]

accumulator = 2
next_value  = 4  → result = 6

accumulator = 6
next_value  = 7  → result = 13

accumulator = 13
next_value  = 3  → result = 16
```

For beginners, supplying an explicit initial value can make the process easier to understand.


### ⚡ Quick check — understand the roles first

Given:


In [ ]:
distances = [12.5, 8.0, 15.5, 4.0]


Suppose we use an initial accumulator of `0`.

Complete the first two conceptual calls:

```text
First call:
accumulator = ______
next_value  = ______
returned value = ______

Second call:
accumulator = ______
next_value  = ______
returned value = ______
```

Only after identifying those values, complete the `reduce()`:


In [ ]:
from functools import reduce

distances = [12.5, 8.0, 15.5, 4.0]

total_distance = reduce(
    lambda accumulator, next_value: __________,
    distances,
    0
)

# Expected:
# 40.0


# 7. Choosing the right pattern

Use this as a quick decision guide:

```text
Do I want to change every element?
            ↓
           map()

Do I want to keep only some elements?
            ↓
         filter()

Do I want to combine everything into one result?
            ↓
          reduce()

Do I want to order elements by a criterion?
            ↓
 sorted(..., key=...)
```

And remember:

> `lambda` is often useful when the helper function is short, local, and not worth naming.


## 🧩 Mini Challenge — combine the patterns

You have delivery data stored as `(city, minutes)` tuples.

Complete the following:

1. Keep only deliveries under 60 minutes using `filter()`.
2. Extract only their city names using `map()`.
3. Calculate the total duration of **all** deliveries using `reduce()`.


In [ ]:
from functools import reduce

deliveries = [
    ("Madrid", 45),
    ("Valencia", 75),
    ("Seville", 55),
    ("Bilbao", 90),
    ("Malaga", 40)
]

fast_deliveries = list(
    filter(lambda delivery: __________, deliveries)
)

city_names = list(
    map(lambda delivery: __________, fast_deliveries)
)

total_minutes = reduce(
    lambda total, delivery: __________,
    deliveries,
    0
)

# Check your results:
# fast_deliveries
# city_names
# total_minutes


# 🚀 Lab Readiness Check

Before starting the Lab, make sure you can answer **yes** to these questions:

- [ ] Can I explain what a lambda function is?
- [ ] Can I decide when `def` is clearer than `lambda`?
- [ ] Can I use a lambda with `sorted(..., key=...)`?
- [ ] Can I use `map()` when every element must be transformed?
- [ ] Can I use a named function with `map()`?
- [ ] Can I use `filter()` when only some elements should remain?
- [ ] Do I understand that `map()` and `filter()` return iterators?
- [ ] Can I import and use `reduce()`?
- [ ] Can I explain what the accumulator represents in `reduce()`?
- [ ] Can I identify whether a problem is a transformation, filter, reduction, or sorting problem?

If yes, you have the tools needed for the Lab.


---

# 📚 Optional — self-learning / reference

The following topics are useful extensions but are **not required for today's Lab**.

## `map()` with multiple iterables

`map()` can receive more than one iterable if the function accepts several arguments.


In [ ]:
quantities = [2, 3, 1]
prices = [10, 5, 20]

totals = list(
    map(lambda quantity, price: quantity * price, quantities, prices)
)

totals


## Initial value in `reduce()`

`reduce()` can receive an optional initial accumulator value:


In [ ]:
from functools import reduce

values = [10, 20, 30]

total = reduce(
    lambda accumulated, value: accumulated + value,
    values,
    100
)

total


Here the accumulation starts from `100` instead of from the first element.

## Named functions vs lambdas

A lambda expression can call another function:


In [ ]:
def normalize_score(score):
    capped = min(score, 100)
    return round(capped / 100, 2)

scores = [72, 88, 105]

normalized = list(
    map(lambda score: normalize_score(score), scores)
)

normalized


This is valid because the lambda itself still contains a single expression.

However, if the lambda is only forwarding one argument to an existing function, the named function can usually be passed directly:

```python
list(map(normalize_score, scores))
```

> **Rule of thumb:** use `lambda` when it makes the code clearer, not merely shorter.
